# 🔄 Lección 9: Transformaciones en Manim

## Contenido de esta lección:
1. **Transform** - Transformación básica entre mobjects
2. **ReplacementTransform** - Transformación con reemplazo de referencia
3. **MoveToTarget** - Animar hacia un estado objetivo
4. **Restore** - Volver al estado guardado
5. **FadeTransform** - Transformación con desvanecimiento
6. **FadeTransformPieces** - Transformación de submobjects con fade
7. **ClockwiseTransform / CounterclockwiseTransform** - Transformaciones con rotación
8. **CyclicReplace** - Intercambio cíclico de posiciones
9. **Swap** - Intercambio de dos mobjects
10. **ApplyFunction** - Aplicar función personalizada
11. **ApplyMatrix** - Transformaciones matriciales
12. **TransformFromCopy** - Transformación desde una copia
13. **ScaleInPlace / ShrinkToCenter** - Transformaciones de escala
14. **Comparativa** - Cuándo usar cada transformación
15. **Ejercicios**

---

In [ ]:
from manim import *
import numpy as np

---
## 1. 🔄 Transform - La Transformación Fundamental

`Transform` es la animación base para convertir un mobject en otro.

### Comportamiento clave:
- El mobject **original se modifica** para verse como el objetivo
- El objeto **original sigue siendo la referencia** en la escena
- El objetivo **nunca se añade** a la escena

### Parámetros importantes:
| Parámetro | Descripción | Default |
|-----------|-------------|---------|
| `path_arc` | Ángulo del arco en radianes | 0 (línea recta) |
| `path_func` | Función que define la trayectoria | None |
| `replace_mobject_with_target_in_scene` | Si reemplazar al final | False |
| `run_time` | Duración de la animación | 1.0 |

In [ ]:
%%manim -qm -v WARNING TransformBasico

class TransformBasico(Scene):
    def construct(self):
        titulo = Text("Transform - Básico", font_size=32).to_edge(UP)
        
        # Crear mobjects
        circulo = Circle(radius=1, color=BLUE, fill_opacity=0.5)
        cuadrado = Square(side_length=2, color=RED, fill_opacity=0.5)
        triangulo = Triangle(color=GREEN, fill_opacity=0.5).scale(1.5)
        
        self.play(Write(titulo))
        self.play(Create(circulo))
        self.wait(0.5)
        
        # Transform: circulo se convierte en cuadrado
        # Pero la variable 'circulo' sigue siendo la referencia
        self.play(Transform(circulo, cuadrado))
        self.wait(0.5)
        
        # Para seguir transformando, usamos 'circulo' (no 'cuadrado')
        self.play(Transform(circulo, triangulo))
        self.wait(0.5)
        
        # Demostrar que 'circulo' es la referencia
        nota = Text("'circulo' sigue siendo la referencia", font_size=20, color=YELLOW)
        nota.to_edge(DOWN)
        self.play(Write(nota))
        
        # Animar el objeto (usando circulo, no triangulo)
        self.play(circulo.animate.shift(RIGHT * 2))
        self.play(circulo.animate.rotate(PI/4))
        
        self.wait()

### 1.1 Transform con path_arc

El parámetro `path_arc` añade una trayectoria curva a la transformación:

In [ ]:
%%manim -qm -v WARNING TransformPathArc

class TransformPathArc(Scene):
    def construct(self):
        titulo = Text("Transform con path_arc", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear círculos en diferentes posiciones
        colores = [RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE]
        angulos = [0, 30, 90, 180, 270, -90]  # en grados
        
        for i, (color, angulo) in enumerate(zip(colores, angulos)):
            # Círculo origen (izquierda)
            origen = Circle(radius=0.4, color=color, fill_opacity=0.8)
            origen.move_to(LEFT * 3 + UP * (2 - i * 0.8))
            
            # Círculo destino (derecha)
            destino = Circle(radius=0.4, color=color, fill_opacity=0.8)
            destino.move_to(RIGHT * 3 + UP * (2 - i * 0.8))
            
            # Etiqueta
            label = Text(f"{angulo}°", font_size=18).next_to(origen, LEFT)
            
            self.add(label)
            self.play(
                Transform(origen, destino, path_arc=angulo * DEGREES),
                run_time=1.5
            )
        
        nota = Text("Ángulos positivos = antihorario, negativos = horario", 
                    font_size=18, color=GRAY).to_edge(DOWN)
        self.play(Write(nota))
        self.wait()

---
## 2. 🔁 ReplacementTransform - Reemplazo de Referencia

`ReplacementTransform` es similar a `Transform`, pero con una diferencia crucial:
**el objeto objetivo reemplaza al original en la escena**.

### Diferencia clave:
```python
# Transform: después debes usar el objeto ORIGINAL
Transform(a, b)
a.animate.shift(UP)  # Correcto

# ReplacementTransform: después debes usar el objeto DESTINO
ReplacementTransform(a, b)
b.animate.shift(UP)  # Correcto
```

In [ ]:
%%manim -qm -v WARNING ReplacementTransformDemo

class ReplacementTransformDemo(Scene):
    def construct(self):
        titulo = Text("ReplacementTransform vs Transform", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # ===== LADO IZQUIERDO: Transform =====
        label_t = Text("Transform", font_size=22, color=BLUE).shift(LEFT * 3 + UP * 2)
        
        a = Circle(color=BLUE, fill_opacity=0.5).scale(0.6).shift(LEFT * 3)
        b = Square(color=RED, fill_opacity=0.5).scale(0.6).shift(LEFT * 3)
        c = Triangle(color=GREEN, fill_opacity=0.5).scale(0.6).shift(LEFT * 3)
        
        self.play(Write(label_t))
        self.play(Create(a))
        
        # Con Transform, siempre usamos 'a'
        self.play(Transform(a, b))
        self.play(Transform(a, c))  # Seguimos usando 'a'
        self.play(a.animate.shift(DOWN))  # Funciona con 'a'
        
        # ===== LADO DERECHO: ReplacementTransform =====
        label_r = Text("ReplacementTransform", font_size=22, color=YELLOW).shift(RIGHT * 3 + UP * 2)
        
        x = Circle(color=BLUE, fill_opacity=0.5).scale(0.6).shift(RIGHT * 3)
        y = Square(color=RED, fill_opacity=0.5).scale(0.6).shift(RIGHT * 3)
        z = Triangle(color=GREEN, fill_opacity=0.5).scale(0.6).shift(RIGHT * 3)
        
        self.play(Write(label_r))
        self.play(Create(x))
        
        # Con ReplacementTransform, la referencia cambia
        self.play(ReplacementTransform(x, y))  # Ahora usamos 'y'
        self.play(ReplacementTransform(y, z))  # Ahora usamos 'z'
        self.play(z.animate.shift(DOWN))  # Funciona con 'z'
        
        # Explicación
        explicacion = VGroup(
            Text("Transform: usa siempre el objeto ORIGINAL", font_size=18, color=BLUE),
            Text("ReplacementTransform: usa el objeto DESTINO", font_size=18, color=YELLOW)
        ).arrange(DOWN, buff=0.3).to_edge(DOWN)
        
        self.play(Write(explicacion))
        self.wait()

---
## 3. 🎯 MoveToTarget - Animar hacia un Estado Objetivo

`MoveToTarget` es muy útil cuando quieres definir múltiples cambios
y animarlos todos a la vez. El flujo es:

1. Llamar `mobject.generate_target()`
2. Modificar `mobject.target` como quieras
3. Animar con `MoveToTarget(mobject)`

In [ ]:
%%manim -qm -v WARNING MoveToTargetDemo

class MoveToTargetDemo(Scene):
    def construct(self):
        titulo = Text("MoveToTarget - Estado Objetivo", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear un círculo
        circulo = Circle(radius=1, color=BLUE, fill_opacity=0.5)
        self.play(Create(circulo))
        self.wait(0.5)
        
        # Paso 1: Generar el target (copia del objeto)
        circulo.generate_target()
        
        # Paso 2: Modificar el target como queramos
        circulo.target.shift(RIGHT * 2)
        circulo.target.scale(0.5)
        circulo.target.set_color(RED)
        circulo.target.set_fill(opacity=1)
        
        # Paso 3: Animar hacia el target
        self.play(MoveToTarget(circulo))
        self.wait(0.5)
        
        # Podemos repetir el proceso
        circulo.generate_target()
        circulo.target.shift(UP * 2 + LEFT * 4)
        circulo.target.scale(3)
        circulo.target.set_color(GREEN)
        circulo.target.rotate(PI/4)
        
        self.play(MoveToTarget(circulo), run_time=1.5)
        
        # Explicación
        pasos = VGroup(
            Text("1. obj.generate_target()", font_size=18),
            Text("2. obj.target.shift/scale/color...", font_size=18),
            Text("3. MoveToTarget(obj)", font_size=18)
        ).arrange(DOWN, aligned_edge=LEFT).to_edge(DOWN, buff=0.5)
        
        for paso in pasos:
            self.play(Write(paso), run_time=0.5)
        
        self.wait()

---
## 4. ↩️ Restore - Volver al Estado Guardado

`Restore` permite volver a un estado previamente guardado con `save_state()`.
Muy útil para animaciones de "antes y después".

In [ ]:
%%manim -qm -v WARNING RestoreDemo

class RestoreDemo(Scene):
    def construct(self):
        titulo = Text("Restore - Volver al Estado Guardado", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear un cuadrado
        cuadrado = Square(side_length=2, color=BLUE, fill_opacity=0.7)
        self.play(Create(cuadrado))
        
        # GUARDAR EL ESTADO ACTUAL
        cuadrado.save_state()
        
        estado_guardado = Text("Estado guardado ✓", font_size=20, color=GREEN)
        estado_guardado.to_edge(DOWN)
        self.play(Write(estado_guardado))
        self.wait(0.5)
        
        # Hacer varias modificaciones
        self.play(cuadrado.animate.shift(RIGHT * 2).rotate(PI/4).scale(0.5))
        self.play(cuadrado.animate.set_color(RED))
        self.play(cuadrado.animate.shift(UP * 2))
        self.wait(0.5)
        
        # RESTAURAR al estado guardado
        self.play(FadeOut(estado_guardado))
        restaurando = Text("Restaurando...", font_size=20, color=YELLOW)
        restaurando.to_edge(DOWN)
        self.play(Write(restaurando))
        
        self.play(Restore(cuadrado))
        
        self.play(
            FadeOut(restaurando),
            Write(Text("¡Restaurado!", font_size=20, color=GREEN).to_edge(DOWN))
        )
        
        self.wait()

---
## 5. 🌫️ FadeTransform - Transformación con Desvanecimiento

`FadeTransform` hace una transición suave con efecto de fade entre dos mobjects.
El objeto origen se desvanece mientras el destino aparece.

### Parámetros:
| Parámetro | Descripción |
|-----------|-------------|
| `stretch` | Si el destino se estira durante la animación (default: True) |
| `dim_to_match` | Dimensión para escalar (0=x, 1=y, 2=z) cuando stretch=False |

In [ ]:
%%manim -qm -v WARNING FadeTransformDemo

class FadeTransformDemo(Scene):
    def construct(self):
        titulo = Text("FadeTransform - Transición Suave", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear rectángulos de origen
        rect1 = Rectangle(width=4, height=1, color=BLUE, fill_opacity=0.8).shift(UP + LEFT * 2)
        rect2 = Rectangle(width=4, height=1, color=GREEN, fill_opacity=0.8).shift(LEFT * 2)
        rect3 = Rectangle(width=4, height=1, color=RED, fill_opacity=0.8).shift(DOWN + LEFT * 2)
        
        # Crear círculos de destino
        circ1 = Circle(radius=0.5, color=YELLOW, fill_opacity=0.8).shift(UP + RIGHT * 2)
        circ2 = Circle(radius=0.5, color=YELLOW, fill_opacity=0.8).shift(RIGHT * 2)
        circ3 = Circle(radius=0.5, color=YELLOW, fill_opacity=0.8).shift(DOWN + RIGHT * 2)
        
        # Etiquetas
        label1 = Text("stretch=True", font_size=16).next_to(rect1, LEFT)
        label2 = Text("stretch=False, dim=0", font_size=16).next_to(rect2, LEFT)
        label3 = Text("stretch=False, dim=1", font_size=16).next_to(rect3, LEFT)
        
        self.play(
            Create(rect1), Create(rect2), Create(rect3),
            Write(label1), Write(label2), Write(label3)
        )
        self.wait(0.5)
        
        # Diferentes tipos de FadeTransform
        self.play(
            FadeTransform(rect1, circ1, stretch=True),
            FadeTransform(rect2, circ2, stretch=False, dim_to_match=0),
            FadeTransform(rect3, circ3, stretch=False, dim_to_match=1),
            run_time=2
        )
        
        nota = Text("FadeTransform reemplaza el objeto como ReplacementTransform", 
                    font_size=18, color=GRAY).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 6. 🧩 FadeTransformPieces - Transformar Submobjects

`FadeTransformPieces` aplica FadeTransform a cada **submobject** 
individualmente. Útil para VGroups donde quieres transiciones individuales.

In [ ]:
%%manim -qm -v WARNING FadeTransformPiecesDemo

class FadeTransformPiecesDemo(Scene):
    def construct(self):
        titulo = Text("FadeTransformPieces vs FadeTransform", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Grupo de origen (3 círculos)
        grupo1 = VGroup(
            Circle(color=RED, fill_opacity=0.8),
            Circle(color=GREEN, fill_opacity=0.8),
            Circle(color=BLUE, fill_opacity=0.8)
        ).arrange(RIGHT, buff=0.5).scale(0.5).shift(UP + LEFT * 3)
        
        # Grupo de destino (3 cuadrados)
        grupo1_dest = VGroup(
            Square(color=RED, fill_opacity=0.8),
            Square(color=GREEN, fill_opacity=0.8),
            Square(color=BLUE, fill_opacity=0.8)
        ).arrange(RIGHT, buff=0.5).scale(0.5).shift(UP + RIGHT * 3)
        
        # Segundo par para comparación
        grupo2 = VGroup(
            Circle(color=RED, fill_opacity=0.8),
            Circle(color=GREEN, fill_opacity=0.8),
            Circle(color=BLUE, fill_opacity=0.8)
        ).arrange(RIGHT, buff=0.5).scale(0.5).shift(DOWN + LEFT * 3)
        
        grupo2_dest = VGroup(
            Square(color=RED, fill_opacity=0.8),
            Square(color=GREEN, fill_opacity=0.8),
            Square(color=BLUE, fill_opacity=0.8)
        ).arrange(RIGHT, buff=0.5).scale(0.5).shift(DOWN + RIGHT * 3)
        
        # Etiquetas
        label1 = Text("FadeTransform", font_size=18).next_to(grupo1, LEFT)
        label2 = Text("FadeTransformPieces", font_size=18).next_to(grupo2, LEFT)
        
        self.play(Create(grupo1), Create(grupo2), Write(label1), Write(label2))
        self.wait(0.5)
        
        # Comparar las dos animaciones
        self.play(
            FadeTransform(grupo1, grupo1_dest),        # Todo el grupo junto
            FadeTransformPieces(grupo2, grupo2_dest),  # Cada pieza individual
            run_time=2
        )
        
        nota = Text("FadeTransformPieces anima cada submobject por separado", 
                    font_size=18, color=YELLOW).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 7. 🔃 ClockwiseTransform / CounterclockwiseTransform

Estas transformaciones mueven los puntos a lo largo de un arco en 
dirección horaria o antihoraria.

In [ ]:
%%manim -qm -v WARNING ClockwiseDemo

class ClockwiseDemo(Scene):
    def construct(self):
        titulo = Text("Clockwise vs Counterclockwise Transform", font_size=26).to_edge(UP)
        self.play(Write(titulo))
        
        # Objetos para ClockwiseTransform
        origen_cw = Circle(color=BLUE, fill_opacity=0.8, radius=0.5).shift(UP + LEFT * 3)
        destino_cw = Square(color=RED, fill_opacity=0.8, side_length=1).shift(UP + RIGHT * 3)
        label_cw = Text("Clockwise", font_size=20).next_to(origen_cw, DOWN)
        
        # Objetos para CounterclockwiseTransform
        origen_ccw = Circle(color=BLUE, fill_opacity=0.8, radius=0.5).shift(DOWN + LEFT * 3)
        destino_ccw = Square(color=RED, fill_opacity=0.8, side_length=1).shift(DOWN + RIGHT * 3)
        label_ccw = Text("Counterclockwise", font_size=20).next_to(origen_ccw, DOWN)
        
        # Flechas de referencia
        flecha_cw = CurvedArrow(LEFT * 2 + UP, RIGHT * 2 + UP, angle=-TAU/3, color=GRAY)
        flecha_ccw = CurvedArrow(LEFT * 2 + DOWN, RIGHT * 2 + DOWN, angle=TAU/3, color=GRAY)
        
        self.play(
            Create(origen_cw), Create(origen_ccw),
            Write(label_cw), Write(label_ccw),
            Create(flecha_cw), Create(flecha_ccw)
        )
        self.wait(0.5)
        
        # Ejecutar transformaciones
        self.play(
            ClockwiseTransform(origen_cw, destino_cw),
            CounterclockwiseTransform(origen_ccw, destino_ccw),
            run_time=2
        )
        
        self.wait()

---
## 8. 🔄 CyclicReplace - Intercambio Cíclico

`CyclicReplace` mueve múltiples objetos de forma cíclica:
- El primero toma el lugar del segundo
- El segundo toma el lugar del tercero
- ... y así sucesivamente
- El último toma el lugar del primero

In [ ]:
%%manim -qm -v WARNING CyclicReplaceDemo

class CyclicReplaceDemo(Scene):
    def construct(self):
        titulo = Text("CyclicReplace - Intercambio Cíclico", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear grupo de figuras
        figuras = VGroup(
            Square(color=RED, fill_opacity=0.8),
            Circle(color=GREEN, fill_opacity=0.8),
            Triangle(color=BLUE, fill_opacity=0.8),
            Star(n=5, color=YELLOW, fill_opacity=0.8)
        ).scale(0.6).arrange(RIGHT, buff=1)
        
        # Etiquetas
        labels = VGroup(
            Text("1", font_size=24),
            Text("2", font_size=24),
            Text("3", font_size=24),
            Text("4", font_size=24)
        )
        for label, fig in zip(labels, figuras):
            label.next_to(fig, DOWN)
        
        self.play(Create(figuras), Write(labels))
        self.wait(0.5)
        
        # Realizar intercambios cíclicos
        for i in range(4):
            info = Text(f"Ciclo {i + 1}: 1→2→3→4→1", font_size=20, color=YELLOW)
            info.to_edge(DOWN)
            self.play(Write(info))
            self.play(CyclicReplace(*figuras), run_time=1.5)
            self.play(FadeOut(info))
        
        self.wait()

---
## 9. ↔️ Swap - Intercambio de Dos Objetos

`Swap` es un caso especial de `CyclicReplace` para exactamente dos objetos.

In [ ]:
%%manim -qm -v WARNING SwapDemo

class SwapDemo(Scene):
    def construct(self):
        titulo = Text("Swap - Intercambio de Dos Objetos", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear dos objetos
        obj1 = VGroup(
            Circle(color=BLUE, fill_opacity=0.8, radius=0.8),
            Text("A", font_size=36, color=WHITE)
        ).shift(LEFT * 2)
        
        obj2 = VGroup(
            Square(color=RED, fill_opacity=0.8, side_length=1.5),
            Text("B", font_size=36, color=WHITE)
        ).shift(RIGHT * 2)
        
        self.play(Create(obj1), Create(obj2))
        self.wait(0.5)
        
        # Intercambiar posiciones con Swap
        for i in range(3):
            self.play(Swap(obj1, obj2), run_time=1)
            self.wait(0.3)
        
        nota = Text("Swap intercambia las posiciones de dos mobjects", 
                    font_size=20, color=GRAY).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 10. ⚡ ApplyFunction - Aplicar Función Personalizada

`ApplyFunction` permite aplicar cualquier función que modifique un mobject.
La función recibe el mobject y debe retornarlo modificado.

In [ ]:
%%manim -qm -v WARNING ApplyFunctionDemo

class ApplyFunctionDemo(Scene):
    def construct(self):
        titulo = Text("ApplyFunction - Función Personalizada", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear un cuadrado
        cuadrado = Square(side_length=2, color=BLUE, fill_opacity=0.7)
        self.play(Create(cuadrado))
        
        # Definir una función personalizada
        def transformacion_compleja(mob):
            mob.scale(0.5)
            mob.rotate(PI/4)
            mob.set_color(RED)
            mob.shift(RIGHT * 2 + UP)
            return mob
        
        # Aplicar la función
        self.play(ApplyFunction(transformacion_compleja, cuadrado))
        self.wait(0.5)
        
        # Otra función
        def ondular(mob):
            mob.scale(2)
            mob.set_color(GREEN)
            mob.shift(LEFT * 4)
            return mob
        
        self.play(ApplyFunction(ondular, cuadrado))
        
        nota = Text("La función recibe el mobject y lo retorna modificado", 
                    font_size=18, color=GRAY).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 11. 📐 ApplyMatrix - Transformaciones Matriciales

`ApplyMatrix` aplica una transformación lineal definida por una matriz.
Muy útil para demostraciones de álgebra lineal.

In [ ]:
%%manim -qm -v WARNING ApplyMatrixDemo

class ApplyMatrixDemo(Scene):
    def construct(self):
        titulo = Text("ApplyMatrix - Transformaciones Matriciales", font_size=26).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear un cuadrado unitario y ejes
        cuadrado = Square(side_length=1.5, color=BLUE, fill_opacity=0.5)
        cuadrado.shift(RIGHT * 0.75 + UP * 0.75)  # Posicionar en primer cuadrante
        
        ejes = Axes(
            x_range=[-3, 3], y_range=[-3, 3],
            x_length=6, y_length=6,
            axis_config={"include_tip": True}
        ).scale(0.8)
        
        self.play(Create(ejes), Create(cuadrado))
        self.wait(0.5)
        
        # Matriz de escala
        matriz_escala = [[2, 0], [0, 0.5]]
        label1 = MathTex(r"\begin{bmatrix} 2 & 0 \\ 0 & 0.5 \end{bmatrix}", font_size=28)
        label1.to_corner(UL).shift(DOWN)
        texto1 = Text("Escala", font_size=20, color=YELLOW).next_to(label1, DOWN)
        
        self.play(Write(label1), Write(texto1))
        self.play(ApplyMatrix(matriz_escala, cuadrado), run_time=1.5)
        self.wait(0.5)
        
        # Volver al original
        self.play(ApplyMatrix([[0.5, 0], [0, 2]], cuadrado))
        self.play(FadeOut(label1), FadeOut(texto1))
        
        # Matriz de rotación (90 grados)
        matriz_rot = [[0, -1], [1, 0]]
        label2 = MathTex(r"\begin{bmatrix} 0 & -1 \\ 1 & 0 \end{bmatrix}", font_size=28)
        label2.to_corner(UL).shift(DOWN)
        texto2 = Text("Rotación 90°", font_size=20, color=GREEN).next_to(label2, DOWN)
        
        self.play(Write(label2), Write(texto2))
        self.play(ApplyMatrix(matriz_rot, cuadrado), run_time=1.5)
        self.wait(0.5)
        
        self.play(FadeOut(label2), FadeOut(texto2))
        
        # Matriz de cizallamiento (shear)
        matriz_shear = [[1, 1], [0, 1]]
        label3 = MathTex(r"\begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}", font_size=28)
        label3.to_corner(UL).shift(DOWN)
        texto3 = Text("Cizallamiento", font_size=20, color=RED).next_to(label3, DOWN)
        
        self.play(Write(label3), Write(texto3))
        self.play(ApplyMatrix(matriz_shear, cuadrado), run_time=1.5)
        
        self.wait()

---
## 12. 📋 TransformFromCopy - Transformar desde una Copia

`TransformFromCopy` crea una copia del origen y la transforma hacia el destino,
manteniendo el original intacto. Útil para mostrar "derivaciones" o "ramificaciones".

In [ ]:
%%manim -qm -v WARNING TransformFromCopyDemo

class TransformFromCopyDemo(Scene):
    def construct(self):
        titulo = Text("TransformFromCopy - Mantener el Original", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Objeto original
        original = Circle(radius=1, color=BLUE, fill_opacity=0.7)
        original.shift(LEFT * 3)
        label_orig = Text("Original", font_size=20).next_to(original, DOWN)
        
        self.play(Create(original), Write(label_orig))
        self.wait(0.5)
        
        # Destinos
        destino1 = Square(side_length=1.5, color=RED, fill_opacity=0.7)
        destino1.shift(RIGHT * 1 + UP * 1.5)
        
        destino2 = Triangle(color=GREEN, fill_opacity=0.7).scale(1.2)
        destino2.shift(RIGHT * 1 + DOWN * 1.5)
        
        destino3 = Star(n=5, color=YELLOW, fill_opacity=0.7)
        destino3.shift(RIGHT * 4)
        
        # TransformFromCopy: el original permanece
        self.play(TransformFromCopy(original, destino1))
        self.wait(0.3)
        
        self.play(TransformFromCopy(original, destino2))
        self.wait(0.3)
        
        self.play(TransformFromCopy(original, destino3))
        
        # Destacar que el original sigue ahí
        self.play(
            original.animate.set_color(GOLD),
            Flash(original, color=GOLD)
        )
        
        nota = Text("El original permanece, se crea una copia que se transforma", 
                    font_size=18, color=YELLOW).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 13. 📏 ScaleInPlace / ShrinkToCenter

Transformaciones especializadas para escalar objetos:
- `ScaleInPlace`: Escala manteniendo el centro
- `ShrinkToCenter`: Reduce el objeto hasta desaparecer en su centro

In [ ]:
%%manim -qm -v WARNING ScaleAnimationsDemo

class ScaleAnimationsDemo(Scene):
    def construct(self):
        titulo = Text("ScaleInPlace y ShrinkToCenter", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # ScaleInPlace
        cuadrado = Square(side_length=2, color=BLUE, fill_opacity=0.7)
        cuadrado.shift(LEFT * 3)
        label1 = Text("ScaleInPlace", font_size=20).next_to(cuadrado, DOWN)
        
        self.play(Create(cuadrado), Write(label1))
        
        # Escalar hacia arriba
        self.play(ScaleInPlace(cuadrado, 1.5))
        self.wait(0.3)
        
        # Escalar hacia abajo
        self.play(ScaleInPlace(cuadrado, 0.5))
        self.wait(0.5)
        
        # ShrinkToCenter
        circulo = Circle(radius=1, color=RED, fill_opacity=0.7)
        circulo.shift(RIGHT * 3)
        label2 = Text("ShrinkToCenter", font_size=20).next_to(circulo, DOWN, buff=0.5)
        
        self.play(Create(circulo), Write(label2))
        self.wait(0.5)
        
        # ShrinkToCenter es un "remover" - el objeto desaparece
        self.play(ShrinkToCenter(circulo))
        
        nota = Text("ShrinkToCenter es un 'remover': el objeto desaparece", 
                    font_size=18, color=YELLOW).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

---
## 14. 📊 Comparativa - Cuándo Usar Cada Transformación

| Transformación | Cuándo usarla | Referencia después |
|----------------|---------------|---------------------|
| `Transform` | Cambiar forma manteniendo misma referencia | Original |
| `ReplacementTransform` | Cambiar forma y referencia | Destino |
| `MoveToTarget` | Múltiples cambios predefinidos | Original |
| `Restore` | Volver a un estado guardado | Original |
| `FadeTransform` | Transición suave con fade | Destino |
| `FadeTransformPieces` | Fade por submobjects | Destino |
| `ClockwiseTransform` | Movimiento en arco horario | Original |
| `CounterclockwiseTransform` | Movimiento en arco antihorario | Original |
| `CyclicReplace` | Intercambio cíclico de posiciones | Originales |
| `Swap` | Intercambio de dos objetos | Originales |
| `ApplyFunction` | Función personalizada de transformación | Original |
| `ApplyMatrix` | Transformación lineal matricial | Original |
| `TransformFromCopy` | Crear copia y transformarla | Destino (copia) |
| `ScaleInPlace` | Escalar manteniendo centro | Original |
| `ShrinkToCenter` | Desaparecer hacia el centro | (removido) |

In [ ]:
%%manim -qm -v WARNING ComparativaCompleta

class ComparativaCompleta(Scene):
    def construct(self):
        titulo = Text("Ejemplo Práctico: Todas las Transformaciones", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear un objeto central
        original = Circle(radius=0.8, color=BLUE, fill_opacity=0.8)
        self.play(Create(original))
        self.wait(0.5)
        
        # Guardar estado
        original.save_state()
        
        # Serie de transformaciones
        transformaciones = [
            ("Transform → Cuadrado", Square(color=RED, fill_opacity=0.8), Transform),
            ("MoveToTarget", None, "target"),
            ("Restore", None, Restore),
            ("ReplacementTransform → Triángulo", Triangle(color=GREEN, fill_opacity=0.8).scale(1.2), ReplacementTransform),
        ]
        
        # Transform
        cuadrado = Square(color=RED, fill_opacity=0.8)
        info1 = Text("1. Transform → Cuadrado", font_size=20, color=YELLOW).to_edge(DOWN)
        self.play(Write(info1))
        self.play(Transform(original, cuadrado))
        self.wait(0.3)
        self.play(FadeOut(info1))
        
        # MoveToTarget
        info2 = Text("2. MoveToTarget (escalar + mover + rotar)", font_size=20, color=YELLOW).to_edge(DOWN)
        self.play(Write(info2))
        original.generate_target()
        original.target.scale(0.5).shift(RIGHT * 2).rotate(PI/4).set_color(PURPLE)
        self.play(MoveToTarget(original))
        self.wait(0.3)
        self.play(FadeOut(info2))
        
        # Restore
        info3 = Text("3. Restore al estado original", font_size=20, color=YELLOW).to_edge(DOWN)
        self.play(Write(info3))
        self.play(Restore(original))
        self.wait(0.3)
        self.play(FadeOut(info3))
        
        # TransformFromCopy
        info4 = Text("4. TransformFromCopy (mantiene original)", font_size=20, color=YELLOW).to_edge(DOWN)
        self.play(Write(info4))
        
        destinos = [
            Square(color=ORANGE).scale(0.6).shift(UR * 2),
            Triangle(color=PINK).scale(0.6).shift(UL * 2),
            Star(n=5, color=TEAL).scale(0.5).shift(DR * 2)
        ]
        
        for dest in destinos:
            self.play(TransformFromCopy(original, dest), run_time=0.7)
        
        self.play(Flash(original, color=BLUE))
        
        self.wait()

---
## 15. 📝 Ejercicios

### Ejercicio 1: Cadena de Transformaciones
Crea una escena donde un círculo se transforme en:
1. Un cuadrado (usando Transform)
2. Un triángulo (usando Transform)
3. Vuelva al estado original (usando Restore)

### Ejercicio 2: Carrusel de Formas
Usa `CyclicReplace` para crear un carrusel con 5 formas diferentes
que roten de posición infinitamente (usa un loop).

### Ejercicio 3: Demostración de Matrices
Crea una escena que demuestre las siguientes transformaciones matriciales:
- Reflexión respecto al eje X
- Reflexión respecto al eje Y
- Escalado no uniforme

### Ejercicio 4: Sistema Solar Simple
Usa `MoveToTarget` para crear una animación donde:
- Un "sol" en el centro permanece fijo
- Varios "planetas" se mueven a posiciones orbitales usando `MoveToTarget`

### Ejercicio 5: Comparación Visual
Crea una escena que muestre lado a lado la diferencia entre:
- `Transform` vs `ReplacementTransform`
- `FadeTransform` vs `Transform`

In [ ]:
# Espacio para resolver los ejercicios

# Ejercicio 1


# Ejercicio 2


# Ejercicio 3


# Ejercicio 4


# Ejercicio 5


---
## 16. 📚 Resumen y Referencia Rápida

### Transformaciones Básicas:
```python
Transform(a, b)                    # a se ve como b, referencia: a
ReplacementTransform(a, b)         # a se ve como b, referencia: b
```

### Transformaciones con Estado:
```python
obj.generate_target()              # Crear target
obj.target.shift(UP).scale(2)      # Modificar target
MoveToTarget(obj)                  # Animar hacia target

obj.save_state()                   # Guardar estado
# ... modificaciones ...
Restore(obj)                       # Volver al estado guardado
```

### Transformaciones con Fade:
```python
FadeTransform(a, b)                # Fade suave
FadeTransform(a, b, stretch=False) # Sin estiramiento
FadeTransformPieces(grupo1, grupo2) # Fade por submobjects
```

### Transformaciones Direccionales:
```python
ClockwiseTransform(a, b)           # Arco horario
CounterclockwiseTransform(a, b)    # Arco antihorario
Transform(a, b, path_arc=PI/2)     # Arco personalizado
```

### Intercambios:
```python
CyclicReplace(a, b, c, d)          # a→b→c→d→a
Swap(a, b)                         # a↔b
```

### Funciones y Matrices:
```python
ApplyFunction(mi_funcion, obj)     # Función personalizada
ApplyMatrix([[2,0],[0,2]], obj)    # Transformación matricial
```

### Otras:
```python
TransformFromCopy(origen, destino) # Copia del origen → destino
ScaleInPlace(obj, factor)          # Escalar en su lugar
ShrinkToCenter(obj)                # Desaparecer al centro
```

---
## 17. 🔗 Recursos y Referencias

### Documentación oficial:
- [Transform](https://docs.manim.community/en/stable/reference/manim.animation.transform.Transform.html)
- [ReplacementTransform](https://docs.manim.community/en/stable/reference/manim.animation.transform.ReplacementTransform.html)
- [MoveToTarget](https://docs.manim.community/en/stable/reference/manim.animation.transform.MoveToTarget.html)
- [FadeTransform](https://docs.manim.community/en/stable/reference/manim.animation.transform.FadeTransform.html)
- [CyclicReplace](https://docs.manim.community/en/stable/reference/manim.animation.transform.CyclicReplace.html)
- [ApplyMatrix](https://docs.manim.community/en/stable/reference/manim.animation.transform.ApplyMatrix.html)
- [Módulo transform completo](https://docs.manim.community/en/stable/reference/manim.animation.transform.html)

### Tutoriales relacionados:
- [Manim Example Gallery](https://docs.manim.community/en/stable/examples.html)
- [Transformations Guide](https://docs.manim.community/en/stable/tutorials/quickstart.html)

### Tips importantes:
1. **Transform vs ReplacementTransform**: La diferencia está en qué objeto usar después
2. **MoveToTarget es muy útil** para animaciones complejas predefinidas
3. **save_state() + Restore()** son perfectos para animaciones de "antes y después"
4. **path_arc** añade dinamismo a cualquier Transform
5. **TransformFromCopy** es ideal para mostrar "ramificaciones" o "derivaciones"

---

**¡Felicidades!** Has completado la Lección 9 sobre Transformaciones.
En la próxima lección aprenderemos sobre Métodos como Animaciones (`.animate`).